In [1]:
import pandas as pd
import numpy as np
import re
from nltk.corpus import stopwords

In [2]:
stop_words = stopwords.words('turkish') #(gereksiz/anlamsız kelimeler)

In [3]:
def preprocess_text(text):
    text = re.sub(r'\W', ' ', text)  # Noktalama işaretlerini kaldır
    text = text.lower()  # Küçük harfe çevir
    text = text.replace("\n", "")  # Yeni satır karakterlerini boşluk ile değiştir
    text = ' '.join([word for word in text.split() if word not in stop_words])  # Stop words çıkar 
    return text

In [4]:
df= pd.read_csv("turkish_movie_sentiment_dataset.csv")


In [5]:
df['comment'] = df['comment'].apply(preprocess_text)


In [6]:
#df['comment'].head(60)

In [7]:
df.isna().sum()

comment      0
film_name    0
point        0
dtype: int64

In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(stop_words=stop_words)
X = vectorizer.fit_transform(df['comment'])

In [9]:
replacement_dict = {
    '4,6': 'Olumlu',
    '3,9': 'Olumlu',
    '3,8':'Olumlu',
    '3,7':'Notr',
    '3,2':'Notr',
    '3,1':'Notr',
    '0,5':'Olumsuz',
    '1,0':'Olumsuz',
    '1,5':'Olumsuz',
    '2,0':'Olumsuz',
    '2,5':'Notr',
    '3,0':'Notr',
    '3,5':'Notr',
    '4,0':'Olumlu',
    '4,5':'Olumlu',
    '5,0':'Olumlu',
}
df['point'] = df['point'].replace(replacement_dict)

In [10]:
y=df['point']
y.unique()


array(['Olumlu', 'Notr', 'Olumsuz'], dtype=object)

In [11]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

In [12]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)
X_train

<66581x222338 sparse matrix of type '<class 'numpy.float64'>'
	with 2194538 stored elements in Compressed Sparse Row format>

In [13]:
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

        Notr       0.56      0.42      0.48      5050
      Olumlu       0.70      0.87      0.78      8687
     Olumsuz       0.72      0.52      0.61      2909

    accuracy                           0.67     16646
   macro avg       0.66      0.60      0.62     16646
weighted avg       0.66      0.67      0.66     16646



In [14]:
comments =["Çok duygulandım. Filmde önümüze sergilenen eserlerin arkasında ne kadar çok emek olduğunu görüyoruz. Karşımıza çıkan engelleri pes etmeden, kaçmadan nasıl aşabileceğimiz anlatılıyor. Bir opera parçasının yaratılması aşaması örneği ile cumhuriyetin kuruluşu için Atatürk ün nasıl çaba göstermiş olduğu bir hikaye gibi sergileniyor. Böyle önemli güç işlerle uğranların özel hayatlarından feragat etmeleri gerektiği de görülüyor. Adnan saygun un eşinden uzaklaşması, Atatürk ün Sofya daki sevgilisini bırakmak zorunda kalması, hala unutamaması gibi. Adnan Saygun un eşinin durumu anlayamayıp, alınganlık göstermesini de Latife hanımın Atatürk ün iş yoğunluğundan benliği ilgiyi göremeyip kızmasına benzettim. Etkileyici bir filmdi bana göre. Tavsiye ederim.",
          "29 Ekim günü bu filmi izledim. Atatürk ve cumhuriyet sevdası ile filme bu anlamlı günde özellikle gittim. Film de maalesef hiç bir duygu hissedemedim. Ne üzüldüm ne gururlandım ne coştum nede ağladım. Konu çok basitti ve sonu baştan belliydi. Özellikle tepki sahneleri çok basit ve kötü senaryolanmış. İyi bir film olmasını çok isterdim ama olmamış.",
"Atatürkü aşık ya da saçma sapan hallerde görmekten bıktık.. Yani senaristler sanirim kendi küçük akılları kadar kurgulayabiliyor atamızı ama bu da saygısızlık.. Atatürkü devrimlerine aşık ve hayalinde kurduğu Türkiye hedefiyle görmek yerine aslı astarı olmayan ilişkilerini devrin aşkı gibi göstermeniz cidden saygısızlık.. begeni kasmak icin popülist kulturunuze daha fazla atamızı kurban etmeseniz mi artık.. orada başkalarının aşk hikayelernii izleyebilirdik.. daha cok saygi duyardım güzel bir film olacakken yazik.. bazı yerlerde amedeus filminden alıntılar yapilmis bir film daha vardı sonradan alintilanan yaratici olun..",
           "Harikaydı, kadro muhteşemdi herkesin emeğine sağlık ancak ilgi oldukça azdı yeteri kadar reklam yapılamamış sanırım üzüldüm",
"""Şuana kadar izlediğim Türk filmleri içinde kesinlikle ilk 3'te olabilecek bi film un hep belli bir ölçüde kalması ve kurgunun iyi olması sayesinde filmi baştan sona hiç sıkılmadan izledim.
Lakin değinmek istediğim başka konu var o da şu ki Deniz Çakır afişe yer alacak kadar filmde göremedim kendisini. Yosi Mizrahi'den bi tık daha fazla rolü vardı o kadar. Sanırım çantayla münasebeti olduğu için başrol sayıldı
Ayrıca kendisini Leyla ile Mecnun da adam akıllı tanımış biri olarak Ali atay filmi kesinlikle kurtaran yükselten en büyük etken olmuş.Komedi de nasıl başarılıysa dram da aynı başarılı iş çıkartmış.Şapka çıkartmamak elde değil.""","""Kötü bir deneyim, zaman kaybı. Kankamın tavsiyesi üzerine izledim.""","""Bu nasıl bir dengesizlik ya, 2011 mayıs oldu hala ortalıkta birşey yok....""","""Ekim 2009da gösterime girmesi mi bekleniyor. 17 gün sonra 2011e giriyoruz ama. Editörler yalış yazmışlar galiba :)""","""bu filmi merakla bekliyorum,ama neden ha bire tarihi değişiyor ,en kısa sürede vizyona girer umarım"""
          ]

In [15]:
comment = vectorizer.transform(comments)

In [16]:
y_pred=model.predict(comment)
y_pred

array(['Olumlu', 'Olumsuz', 'Olumsuz', 'Olumlu', 'Olumlu', 'Olumsuz',
       'Olumlu', 'Notr', 'Olumlu'], dtype=object)